In [ ]:
import os
import sys

# Standard interactive replacement for the 'parent directory' hack
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

from PPRCalculator import PPRCalculator

import json

from dataclasses import dataclass

In [112]:
@dataclass
class SpeciesGroup:
    group_name: str
    group_seq: int
    biomass: float
    pb: float
    qb: float
    ee: float
    M0b: float
    tl: float
    flow_to_det: float
    ge: float
    gs: float
    respiration: float
    biomass_accum: float
    biomass_accum_rate: float
    immigration: float
    emigration: float
    export: float
    trophic_info: str
    detritus_import: float
    diet_import: float
    taxon_descr: str

    @classmethod
    def from_dict(cls, data: dict):
        """Factory method to create an instance from a dictionary."""
        return cls(**data)

    def __str__(self):
        return f"[{self.model_number}] '{self.model_name}' {self.model_country} ({self.model_year}) -> {self.group_seq}: {self.group_name} | tl: {self.tl}, taxons included: {[self.taxons_included[i]['taxon_name'] + ' (' + self.taxons_included[i]['AphiaID'] + ')' for i in range(len(self.taxons_included))]}"
    
    def to_df_row(self):
        dct = self.__dict__.copy()
        return pd.DataFrame([dct])


In [147]:
class ModelData:

    def __init__(
            # self, species_groups: list[SpeciesGroup], 
            # model_number, model_name, model_country, model_year, lme
            self, json_filepath
            ):

        self.groups_data: pd.DataFrame
        self.groups_taxons: dict[int, list[dict]]  # group_seq: taxons_list
        
        self.data_json = ModelData.load_json_dict(json_filepath)
        species_groups = ModelData.get_species_groups(self.data_json)
        self.groups_data, diet_import = ModelData.get_groups_df(species_groups)

        self.lme = 13 # lme
        self.model_number = 10000 + self.lme # model_number
        self.model_name = 'Humboldt Current South' # model_name
        self.model_country = '?' # model_country
        self.model_year = 1999 # model_year
        
        self.seq2name = ModelData.get_seq2name(self.data_json)
        self.name2seq = {v: k for k, v in self.seq2name.items()}
        import_seq = max(self.seq2name)

        self.groups_data.loc[import_seq] = np.nan
        self.groups_data.loc[import_seq, 'group_name'] = 'diet_import'
        self.groups_data.loc[import_seq, 'trophic_info'] = 'Import'
        self.groups_data.loc[import_seq, 'tl'] = 1.0
        self.groups_data.loc[import_seq, 'respiration'] = 0.0
        diet_import.loc[import_seq] = 0

        # DC and Detritus fate:
        DC, det_fate = ModelData.get_DC(self.data_json)
        DC.loc[import_seq] = 0  # add import data to DC
        DC[import_seq] = diet_import  # add import data to DC
        self.DC = DC.sort_index(ascending=False).sort_index(axis=1, ascending=False)
        
        det_fate.loc[import_seq] = 0  # add diet_import to det_fate
        det_fate[import_seq] = 0  # add diet_import to det_fate
        det_fate.loc[self.groups_data['trophic_info']=='Import', self.groups_data['trophic_info']=='DET'] = 1
        self.det_fate = det_fate.sort_index(ascending=False).sort_index(axis=1, ascending=False)        
    
    @classmethod
    def load_json_dict(cls, filepath):
        """Load a JSON file and return as dictionary."""
        with open(filepath, 'r') as json_file:
            return json.load(json_file)
    
    @classmethod
    def get_seq2name(cls, model):
        """Get mapping from group sequence number to group name."""
        if not isinstance(model, dict):
            return None
        
        groups = model.get('group', {})
        if len(groups) == 0:
            return None
        
        seq2name = {int(g["group_seq"]): g["group_name"] for g in groups}
        import_seq = max(seq2name) + 1
        seq2name[import_seq] = 'diet_import'

        return seq2name

    @classmethod
    def get_DC(cls, model_json_dict):
        """Get diet composition (DC) and detritus fate matrices."""
        if not isinstance(model_json_dict, dict):
            return None
        else:
            model = model_json_dict
        
        groups = model.get('group', {})
        if len(groups) == 0:
            return None
            
        DC_dict = {}
        detritus_fate_dict = {}

        for g in groups:
            diet_descr = g.get('diet_descr', {})
            diet_descr = diet_descr if diet_descr else {}
            diet = diet_descr.get('diet', None)
            if not diet:
                DC_dict[int(g['group_seq'])] = {int(g['group_seq']): None}
                detritus_fate_dict[int(g['group_seq'])] = {int(g['group_seq']): None}
            else:
                diet = diet if isinstance(diet, list) else [diet]
                DC_dict[int(g['group_seq'])] = {int(float(d['prey_seq'])): float(d['proportion']) for d in diet}
                detritus_fate_dict[int(g['group_seq'])] = {int(float(d['prey_seq'])): float(d['detritus_fate']) for d in diet}
        
        # add missing columns:
        DC = pd.DataFrame.from_dict(DC_dict, orient='index').fillna(0)
        for g in groups:
            if int(g['group_seq']) not in DC.columns:
                DC[int(g['group_seq'])] = 0

        DC = DC.sort_index().sort_index(axis=1)

        detritus_fate = pd.DataFrame.from_dict(detritus_fate_dict, orient='index').fillna(0)
        detritus_fate = detritus_fate.sort_index().sort_index(axis=1)

        return DC, detritus_fate
    
    @classmethod
    def get_species_groups(cls, model_json):
        groups = model_json.get('group', {})
        if len(groups) == 0:
            return []
        species_groups = []
        for j in range(len(groups)):
            group_diet_data = groups[j]
            group_info = {
                "group_name": group_diet_data["group_name"],
                "group_seq": int(group_diet_data["group_seq"]),
                "biomass": float(group_diet_data["biomass"]) if group_diet_data.get("biomass", np.nan) != "-9999" else np.nan,
                "pb": float(group_diet_data["pb"]) if group_diet_data.get("pb", np.nan) != "-9999" else np.nan,
                "qb": float(group_diet_data["qb"]) if group_diet_data.get("qb", np.nan) != "-9999" else np.nan,
                "ee": float(group_diet_data["ee"]) if group_diet_data.get("ee", np.nan) != "-9999" else np.nan,
                "M0b": float(group_diet_data["other_mort"]) if group_diet_data.get("other_mort") != "-9999" else np.nan,
                "ge": float(group_diet_data.get("ge", np.nan)) if group_diet_data.get("ge", np.nan) != "-9999" else np.nan,
                "gs": float(group_diet_data.get("gs", np.nan)) if group_diet_data.get("gs", np.nan) != "-9999" else np.nan,
                "respiration": float(group_diet_data.get("respiration", np.nan)) if group_diet_data.get("respiration", np.nan) != "-9999" else np.nan,
                "immigration": float(group_diet_data.get("immigration", np.nan)) if group_diet_data.get("immigration", np.nan) != "-9999" else np.nan,
                "emigration": float(group_diet_data.get("emigration", np.nan)) if group_diet_data.get("emigration", np.nan) != "-9999" else np.nan,
                "biomass_accum": float(group_diet_data.get("biomass_accum", np.nan)) if group_diet_data.get("biomass_accum", np.nan) != "-9999" else np.nan,
                "biomass_accum_rate": float(group_diet_data.get("biomass_accum_rate", np.nan)) if group_diet_data.get("biomass_accum_rate", np.nan) != "-9999" else np.nan,
                "export": float(group_diet_data.get("export", np.nan)) if group_diet_data.get("export", np.nan) != "-9999" else np.nan,
                "trophic_info": (lambda x: ["Regular", "PP", "DET"][int(x)])(float(group_diet_data["pp"])),
                "detritus_import": float(group_diet_data.get("detritus_import", 0)) if group_diet_data["pp"] == "2" else 0,
                "diet_import": float(group_diet_data.get("diet_imp", np.nan)) if group_diet_data.get("diet_imp", np.nan) != "-9999" else np.nan,
                "taxon_descr": group_diet_data["taxon_descr"],
                "tl": np.nan,
            }
            
            # flow to det = flow from unassimilated food + flow from other mortality
            group_info['M0b'] = group_info['pb'] * (1-group_info['ee'])
            flow_from_food = group_info["biomass"] * group_info["qb"] * group_info["gs"]
            flow_from_bodies = group_info["biomass"] * group_info["M0b"]
            group_info['flow_to_det'] = flow_from_food + flow_from_bodies

            species_groups.append(SpeciesGroup.from_dict(group_info))

        return species_groups

    @classmethod
    def get_groups_df(cls, species_groups):
        df_rows = [species_groups[i].to_df_row() for i in range(len(species_groups))]
        df = pd.concat(df_rows, ignore_index=True)
        df = df.replace("-9999", np.nan).replace(-9999, np.nan)

        df = df.rename(columns={  # change names
            'export': 'catch',
            })
        
        df['p'] = df['pb'] * df['biomass']  # production
        df['q'] = df['qb'] * df['biomass']  # consumption
        df['M0'] = df['p'] * (1-df['ee'])  # other mortality
        df['net_migration'] = df['emigration'] - df['immigration']  # net migration
        
        # production*EE = catch + predation + biomass_accum + net_migration:
        df['predation'] = df['p'] * df['ee'] - (df['catch'] + df['biomass_accum'] + df['net_migration'])
        df['egestion'] = df['q'] * df['gs']

        df['flow_to_det'] = df['egestion'] + df['M0']

        cols_to_return = ['group_name', 'trophic_info', 'taxon_descr', 'tl', 'ge', 'ee', 'catch', 
            'biomass', 'pb', 'qb', 'p', 'q', 'predation', 'M0', 'gs', 'egestion', 'respiration', 'biomass_accum', 'emigration', 'immigration', 'net_migration',
            'flow_to_det', 'detritus_import'
        ]

        groups_data = df.set_index('group_seq')[cols_to_return]
        groups_data = groups_data.sort_index(ascending=False)
        diet_import = df.set_index('group_seq')['diet_import'].sort_index(ascending=False)

        return groups_data, diet_import

In [136]:
model_data = ModelData('real_models/EwE_jsons/LME13.json')

In [141]:
model_data.groups_data

,group_name,trophic_info,taxon_descr,tl,ge,ee,catch,biomass,pb,qb,...,M0,gs,egestion,respiration,biomass_accum,emigration,immigration,net_migration,flow_to_det,detritus_import
group_seq,,,,,,,,,,,,,,,,,,,,,
15,Detritus,Regular,All organic matter not included in another fun...,NaN,NaN,0.02,0.000,NaN,NaN,NaN,...,NaN,0.0,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.0
14,Sea lions,Regular,Otaria byronia,NaN,NaN,0.00,0.000,0.035,0.20,14.36,...,0.007000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.007000,0.0
13,Southern hake,Regular,Merluccius australis,NaN,NaN,0.66,0.125,3.140,0.19,0.68,...,0.202844,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.202844,0.0
12,Skates,Regular,Zearaja chilensis,NaN,NaN,0.09,0.000,0.073,0.15,1.24,...,0.009964,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.009964,0.0
11,Kingklip,Regular,Genypterus blacodes,NaN,NaN,0.21,0.013,0.197,0.39,1.40,...,0.060696,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.060696,0.0
10,Southern blue whiting,Regular,Micromesistius australis,NaN,NaN,0.91,0.011,1.400,0.42,3.60,...,0.052920,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.052920,0.0
9,Hoki (a),Regular,Macruronus magellanicus ≥ 3 years old,NaN,NaN,0.04,0.056,2.722,0.70,3.11,...,1.829184,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.829184,0.0
8,Hoki (j),Regular,Macruronus magellanicus < 3 years old,NaN,NaN,0.81,0.006,2.927,1.20,6.24,...,0.667356,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.667356,0.0
7,Other demersal fish,Regular,Seriolella cearulea and Seriolella punctata,NaN,NaN,0.95,0.000,0.774,0.70,3.50,...,0.027090,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.027090,0.0


In [ ]:
def from_json(cls, filepath, underdetermined=False, zero_catch=True, zero_biomass_accum=True, default_gs=True, weight_flow=1.0, weight_guess=1.0):
    pass